In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

In [ ]:
df = pd.read_csv('mes_donnees_normalisées.csv', sep=';')
df.columns = df.columns.str.strip()
#On remplace les valeurs manquantes de P1_reel, V et Lq_reel par la première valeur non nulle de chaque essai selon les conseils de l'encadrante
#Cela permet de conserver d'avantage de lignes avant le dropna()
p1_v0_par_essai = df[df['V'] == 0].groupby('essai')['P1_reel'].first()
df['P1_reel'] = df['essai'].map(p1_v0_par_essai).fillna(df['P1_reel'])

lq_valide_par_essai = df.dropna(subset=['Lq_reel']).groupby('essai')['Lq_reel'].first()
df['Lq_reel'] = df['essai'].map(lq_valide_par_essai).fillna(df['Lq_reel'])
df.loc[df['Lt'] == 'R', 'Lt'] = 0
df.loc[df['Lt'] == 'G', 'Lt'] = 1


In [3]:
#Préparation des données 

# Séparation des features et de la target
df_ut = df[['P1_reel','Lq_reel','P4','E2','E1','E2.1','E2.2','E2.3','E2.4','S_moy','sigma_S']]
df_ut = df_ut.dropna()  # Supprimer les lignes avec des valeurs manquantes
X1 = df_ut.drop(columns=['S_moy','sigma_S'])  # Features
X1['E2_moy'] = X1[['E2.1','E2.2','E2.3','E2.4']].mean(axis=1)  # Moyenne des colonnes E2.1, E2.2, E2.3, E2.4
X1.drop(columns=['E2.1','E2.2','E2.3','E2.4'], inplace=True)  # Supprimer les colonnes E2.1, E2.2, E2.3, E2.4
y1 = df_ut[['S_moy']]  # Target

X_train, X_test, y_train, y_test = train_test_split(X1, y1, test_size=0.2, random_state=42)

In [4]:
#Préparation des données 

# Séparation des features et de la target
df_ut = df[['P1_reel','Lq_reel','P4','E2','E1','S_moy','sigma_S']]
df_ut = df_ut.dropna()  # Supprimer les lignes avec des valeurs manquantes
X1 = df_ut.drop(columns=['S_moy','sigma_S'])  # Features
y1 = df_ut[['S_moy']]  # Target
print(X1)

X_train, X_test, y_train, y_test = train_test_split(X1, y1, test_size=0.2, random_state=42)

       P1_reel   Lq_reel        P4        E2        E1
0     0.531993  0.139073  0.646018  0.762007  0.508681
1     0.531993  0.139073  0.646018  0.599283  0.480526
2     0.531993  0.139073  0.646018  0.675269  0.528390
3     0.531993  0.139073  0.646018  0.620072  0.481933
4     0.531993  0.139073  0.646018  0.750538  0.535429
...        ...       ...       ...       ...       ...
2811  0.957952  0.685430  1.000000  0.693190  0.345378
2812  0.957952  0.685430  1.000000  0.827240  0.381980
2813  0.957952  0.685430  1.000000  0.809319  0.383388
2814  0.957952  0.685430  1.000000  0.980645  0.312999
2815  0.957952  0.685430  1.000000  0.670968  0.308775

[2216 rows x 5 columns]


In [5]:
# Modèle Random Forest
rf = RandomForestRegressor(
    n_estimators=200,   # nombre d'arbres
    max_depth= 8,    # profondeur illimitée
    random_state=42,
    n_jobs=-1          # utilise tous les cœurs CPU
)

# Entraînement
rf.fit(X_train, y_train)

# Prédictions
y_pred = rf.predict(X_test)
y_pred = pd.Series(y_pred, index=y_test.index)

# Métriques
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f"R2: {r2:.06f}")
print(f"MSE: {mse:.06f}")
print()

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse) # La racine carrée pour retrouver l'unité physique de E1

print(f"Erreur Absolue Moyenne (MAE) : {mae:.2f}")
print(f"Racine de l'Erreur Quadratique Moyenne (RMSE) : {rmse:.2f} ")
print("\n")

print("--- 4. Importance des Paramètres ---")
# On récupère le classement d'importance des variables calculé par le modèle
importances = rf.feature_importances_
for col, imp in sorted(zip(X1.columns, importances), key=lambda x: x[1], reverse=True):
    print(f"{col:<10}: {imp*100:.1f}%")


R2: 0.998293
MSE: 0.000091

Erreur Absolue Moyenne (MAE) : 0.00
Racine de l'Erreur Quadratique Moyenne (RMSE) : 0.01 


--- 4. Importance des Paramètres ---
P4        : 65.9%
Lq_reel   : 26.2%
P1_reel   : 4.5%
E1        : 3.4%
E2        : 0.0%


/opt/miniconda3/lib/python3.13/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
